
# 05 — Circos plots of NMF components per location (site)

**What this does**
- Reads `W_samples.csv`, `H_profiles_normalized.csv`, and `component_labels.csv` from  
  `C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\`
- For each **site_id**, draws a **pycirclize** plot with:
  - one **sector per component** (K=4)
  - **outer track**: component weight time-series (per date at that site)
  - **inner track**: top-`TOP_MUTS` mutations (bars) of that component (global)
- Saves PNGs under `...\results\eda\nnmf_poisson\circos\`.


In [2]:
# %% [markdown]
# Circos — 4 lineages (sectors) per site
# Outer track = all per-mutation AF time-series (scatter, 90→100)
# Inner track = all mutation NNMF activation bars (60→88)
# Middle (= chords) = shared mutations across lineages (52–58)
# Small inner hole (8%)
# ------------------------------------------------------------

from __future__ import annotations
import re
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pycirclize import Circos

# ---------------- Paths ----------------
REPO = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PATH_SNV = REPO / "results/preprocessing/tables/feature_store_snv.csv"
PATH_SIG = REPO / "results/preprocessing/tables/feature_store_signatures.csv"
PATH_H   = REPO / "results/eda/nnmf_poisson/H_profiles_normalized.csv"
OUT_DIR  = REPO / "results/eda/nnmf_poisson/circos_lineage_chords"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Config (radial in %) ----------------
FIGSIZE   = (12, 12)
DPI       = 300
LINEAGES  = ["B.1.1.7", "B.1.351", "P.1", "B.1.617.2"]
MAX_LEGEND = 80

R_INNER_HOLE = 1
R_CHORD_IN   = 12
R_CHORD_OUT  = 25
R_BAR_IN     = 60
R_BAR_OUT    = 78
R_SCAT_IN    = 80
R_SCAT_OUT   = 100

LINEAGE_COLOR = {
    "B.1.1.7":   "#002147",
    "B.1.351":   "#2c5a7b",
    "P.1":       "#4b7da1",
    "B.1.617.2": "#9fc0dc",
}

# ---------------- Load ----------------
snv = pd.read_csv(PATH_SNV, parse_dates=["date"])
sig = pd.read_csv(PATH_SIG)
Hdf = pd.read_csv(PATH_H)

def canon(x):
    if pd.isna(x): return ""
    s = str(x).strip().upper().replace(" ", "")
    return re.sub(r"[^A-Z0-9:_\-]", "", s)

snv["site_id"]  = snv["site_id"].astype(str)
snv["mutation"] = snv["mutation"].astype(str).map(canon)
sig["mutation"] = sig["mutation"].astype(str).map(canon)
sig["lineage"]  = sig["lineage"].astype(str)
Hdf["mutation"] = Hdf["mutation"].astype(str).map(canon)

COMP_COLS = [c for c in Hdf.columns if c.lower().startswith("comp")]
if not COMP_COLS:
    raise RuntimeError("No 'Comp *' columns in H_profiles_normalized.csv")
H = Hdf.set_index("mutation")[COMP_COLS].fillna(0.0)
H["activation"] = H.max(axis=1)

# Aggregate AF per site/date/mutation
snv = snv[snv["coverage"] > 0].copy()
snv_g = (snv.groupby(["site_id","date","mutation"])[["count","coverage"]]
           .sum(min_count=1).reset_index())
snv_g["af"] = (snv_g["count"]/snv_g["coverage"]).clip(0,1)

SITES = sorted(snv_g["site_id"].unique())

# palettes
def build_palette(keys):
    cmap = plt.colormaps["tab20"]
    return {k: cmap(i % cmap.N) for i,k in enumerate(sorted(keys))}

ALL_MUTS = sorted(H.index.unique())
MUT_COLOR = build_palette(ALL_MUTS)

# ---------------- Helpers ----------------
def safe(x): return "".join(ch if ch.isalnum() else "_" for ch in str(x))

def site_series(df: pd.DataFrame):
    d = df.dropna(subset=["date"]).copy()
    dates = sorted(pd.to_datetime(d["date"]).unique())
    idx = {dt:i for i,dt in enumerate(dates)}
    T=len(dates)
    out={}
    for m,g in d.groupby("mutation", sort=False):
        y=np.zeros(T, dtype=float)
        for _,r in g.iterrows():
            y[idx[pd.to_datetime(r["date"])]] = float(r["af"])
        out[m]=y
    return dates,out

# ---------------- Renderer ----------------
def circos_for_site(site):
    dsite = snv_g[snv_g["site_id"]==site]
    if dsite.empty:
        print(f"[skip] {site}")
        return
    dates,series = site_series(dsite)
    T=len(dates)
    if T==0: 
        print(f"[skip] {site} (no dates)")
        return

    # per-lineage mutation lists (only those present in H)
    lin2muts = {}
    for lin in LINEAGES:
        muts = sig.loc[sig["lineage"]==lin,"mutation"].unique().tolist()
        muts = [m for m in muts if m in H.index]
        # stable order: by activation desc
        if muts:
            muts = list(H.loc[muts, "activation"].sort_values(ascending=False).index)
        lin2muts[lin] = muts

    # sector sizes = #muts that we actually plot for that lineage (>=1)
    sector_sizes = {lin: max(1, len(lin2muts[lin])) for lin in LINEAGES}

    circ = Circos(sector_sizes, space=8)
    circ.inner_radius = R_INNER_HOLE/100.0  # small inner hole

    # time mapper for each sector length L: centers at (k+0.5)*L/T within [0, L)
    def time_x(L, T):
        return (np.arange(T, dtype=float)+0.5) * (L/max(T,1))

    for sector in circ.sectors:
        lin = sector.name
        color = LINEAGE_COLOR.get(lin, "#333333")
        muts_lin = lin2muts[lin]
        L = sector.size

        # --- Outer track (scatter of AF for ALL muts, big ring 90→100) ---
        tr_ts = sector.add_track((R_SCAT_IN, R_SCAT_OUT), r_pad_ratio=0.02)
        tr_ts.axis()
        xt = time_x(L, T)
        for m in muts_lin:
            y = series.get(m)
            if y is None or not np.any(y>0): 
                continue
            # IMPORTANT: fix scaling so markers use same 0..1 AF scale
            tr_ts.scatter(xt, y, s=9, vmin=0.0, vmax=1.0,
                          color=MUT_COLOR.get(m, "#666"), alpha=0.60)

        sector.text(lin, r=R_SCAT_OUT+2, size=12, color=color, weight="bold")

        # optional sparse month ticks
        if T>=6:
            step = max(1, T//12)
            idx = np.arange(0, T, step, dtype=int)
            labels = [pd.to_datetime(dates[t]).strftime("%b") for t in idx]
            tr_ts.xticks((idx + 0.5) * (L / max(T,1)), labels, label_size=8, tick_length=0.8)

        # --- Inner track (NNMF bars for ALL signature muts, big ring 60→88) ---
        tr_bar = sector.add_track((R_BAR_IN, R_BAR_OUT), r_pad_ratio=0.02)
        tr_bar.axis()
        if muts_lin:
            vals = H.reindex(muts_lin)["activation"].fillna(0.0).to_numpy(float)
            vmax = max(vals.max(), 1e-12)
            vals = vals / vmax   # normalize to 0..1 for this ring
            xbar = np.arange(len(muts_lin), dtype=float) + 0.5
            tr_bar.bar(xbar, vals, 
                       color=[MUT_COLOR.get(m, "#666") for m in muts_lin],
                       lw=0.0, alpha=0.98)

    # --- Chords in middle band (52–58): link the same mutation across lineages ---
    window = 0.80  # angular window around each mutation index
    for mut in ALL_MUTS:
        present = [lin for lin in LINEAGES if mut in lin2muts[lin]]
        if len(present) < 2:
            continue
        # choose a consistent color per mutation
        c = MUT_COLOR.get(mut, "#b3c2df")
        for a,b in combinations(present, 2):
            sa = next(s for s in circ.sectors if s.name==a)
            sb = next(s for s in circ.sectors if s.name==b)
            ia = lin2muts[a].index(mut)
            ib = lin2muts[b].index(mut)
            startA = ia + 0.5 - window/2
            endA   = ia + 0.5 + window/2
            startB = ib + 0.5 - window/2
            endB   = ib + 0.5 + window/2
            # correct API: specify (name, start, end) for each side, and radii
            circ.link((a, startA, endA),
                      (b, startB, endB),
                      r1=R_CHORD_IN/100.0, r2=R_CHORD_OUT/100.0,
                      fc=c, ec="none", alpha=0.40)

    # --- Title & legend (way below) ---
    fig = circ.plotfig(figsize=FIGSIZE)
    fig.suptitle(str(site), y=0.97, fontsize=20, color="#002147", fontweight="bold")

    uniq = [m for m in ALL_MUTS if any(m in v for v in lin2muts.values())][:MAX_LEGEND]
    handles=[Line2D([0],[0],marker="o",ls="",color=MUT_COLOR[m],markersize=6) for m in uniq]
    labels=uniq
    plt.subplots_adjust(bottom=0.23)
    fig.legend(handles, labels,
               loc="lower center", bbox_to_anchor=(0.5, -0.10),
               ncol=min(20, max(2, int(np.ceil(len(labels)/2)))),
               frameon=True, fancybox=True, framealpha=0.95,
               borderpad=0.7, handletextpad=0.5, columnspacing=1.0, fontsize=7.5)

    out=OUT_DIR/f"circos_{safe(site)}.png"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"[ok] {site} → {out}")

# ---------------- Run ----------------
for s in SITES:
    circos_for_site(s)
print(f"\n✅ Lineage-level Circos saved to {OUT_DIR}")


[ok] BE - Laupen - ARA Sensetal → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_BE___Laupen___ARA_Sensetal.png
[ok] GR - Chur - ARA Chur → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_GR___Chur___ARA_Chur.png
[ok] SG - Altenrhein - ARA Altenrhein → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_SG___Altenrhein___ARA_Altenrhein.png
[ok] TI - Lugano - CDA Lugano → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_TI___Lugano___CDA_Lugano.png
[ok] VD - Lausanne - STEP Vidy → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_VD___Lausanne___STEP_Vidy.png
[ok] ZH - ZÃ¼rich - ARA WerdhÃ¶lzli → C:\Users\osoom\OneDrive\Desktop\ssssss

In [ ]:
# %% [markdown]
# Circos — 4 lineages (sectors) per site
# Outer track = ALL per-mutation AF time-series (scatter, r=93→100)
# Inner track = ALL signature mutation NNMF activations (bars, r=68→92)
# Middle band  = chords (two layers):
#   A) shared signature aliases (r=55→60)
#   B) high AF co-activation correlations (r=60→66)
# Small inner hole (r=3)
# ------------------------------------------------------------

from __future__ import annotations
import re
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pycirclize import Circos

# ---------------- Paths ----------------
REPO = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PATH_SNV = REPO / "results/preprocessing/tables/feature_store_snv.csv"
PATH_SIG = REPO / "results/preprocessing/tables/feature_store_signatures.csv"
PATH_H   = REPO / "results/eda/nnmf_poisson/H_profiles_normalized.csv"
OUT_DIR  = REPO / "results/eda/nnmf_poisson/circos_lineage_chords"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Visual config (radial units 0..100) ----------------
FIGSIZE   = (12, 12)
DPI       = 300
LINEAGES  = ["B.1.1.7", "B.1.351", "P.1", "B.1.617.2"]
MAX_LEGEND = 80

R_INNER_HOLE = 3

# chord band split into two thin layers so you can see both types
R_ALIAS_IN, R_ALIAS_OUT = 55, 60      # shared-alias chords (lower layer)
R_CORR_IN,  R_CORR_OUT  = 60, 66      # correlation chords (upper layer)

R_BAR_IN,   R_BAR_OUT   = 68, 92      # inner bars
R_SCAT_IN,  R_SCAT_OUT  = 93, 100     # outer scatter

LINEAGE_COLOR = {
    "B.1.1.7":   "#002147",
    "B.1.351":   "#2c5a7b",
    "P.1":       "#4b7da1",
    "B.1.617.2": "#9fc0dc",
}

# ---------------- Chord density / logic ----------------
# Shared-alias chords:
MAX_ALIAS_LINKS_PER_PAIR = 200   # cap per lineage pair to avoid overdraw

# Correlation chords (AF time-series based):
AF_PRESENT_THRESH    = 0.01      # AF must exceed this to count as "present"
MIN_PRESENT_FRAC     = 0.10      # each mutation must be present on ≥10% of dates
MIN_COMMON_DATES     = 6         # both series must have at least this many non-NaN overlaps
CORR_THRESHOLD       = 0.6       # Pearson r threshold
MAX_CORR_LINKS_PER_PAIR = 40     # cap per lineage pair (keeps figure readable)

CORR_COLOR = "#77c18d"           # color for correlation chords (green-ish)
ALIAS_FALLBACK_COLOR = "#b3c2df" # color for alias chords if no mutation color found
CHORD_ALPHA = 0.50               # base alpha; correlation chords scaled by r

# ---------------- Canonicalization ----------------
def canon_exact(x: str) -> str:
    """Strict canonical form for plotting labels & colors."""
    if pd.isna(x): return ""
    s = str(x).strip().upper().replace(" ", "")
    s = re.sub(r"^(AA:|NUC:)", "", s)      # drop AA:/NUC:
    s = re.sub(r"[^A-Z0-9:_\-]", "", s)    # keep limited charset
    return s

def alias_key(x: str) -> str:
    """
    Alias for chord matching: token after the last colon.
    'AA:S:N501Y'/'S:N501Y' → 'N501Y'; 'C913T' → 'C913T'
    """
    s = canon_exact(x)
    return s.split(":")[-1] if ":" in s else s

# ---------------- Load ----------------
snv = pd.read_csv(PATH_SNV, parse_dates=["date"])
sig = pd.read_csv(PATH_SIG)
Hdf = pd.read_csv(PATH_H)

snv["site_id"]  = snv["site_id"].astype(str)
snv["mutation"] = snv["mutation"].astype(str).map(canon_exact)

sig["mutation"] = sig["mutation"].astype(str).map(canon_exact)
sig["lineage"]  = sig["lineage"].astype(str)

Hdf["mutation"] = Hdf["mutation"].astype(str).map(canon_exact)
COMP_COLS = [c for c in Hdf.columns if c.lower().startswith("comp")]
if not COMP_COLS:
    raise RuntimeError("No 'Comp *' columns in H_profiles_normalized.csv")
H = Hdf.set_index("mutation")[COMP_COLS].fillna(0.0)
H["activation"] = H.max(axis=1)

# Aggregate AF per site/date/mutation
snv = snv[snv["coverage"] > 0].copy()
snv_g = (snv.groupby(["site_id","date","mutation"])[["count","coverage"]]
           .sum(min_count=1).reset_index())
snv_g["af"] = (snv_g["count"]/snv_g["coverage"]).clip(0,1)

SITES = sorted(snv_g["site_id"].unique())

# palettes
def build_palette(keys):
    try:
        cmap = plt.colormaps["tab20"]
    except Exception:
        cmap = plt.get_cmap("tab20")
    keys_sorted = sorted(set(keys))
    return {k: cmap(i % cmap.N) for i,k in enumerate(keys_sorted)}

ALL_MUTS = sorted(H.index.unique())    # only color things we can bar-plot (exist in H)
MUT_COLOR = build_palette(ALL_MUTS)

# ---------------- Helpers ----------------
def safe(x): return "".join(ch if ch.isalnum() else "_" for ch in str(x))

def site_series(df: pd.DataFrame):
    """Return (dates, {mutation -> AF vector over dates})."""
    d = df.dropna(subset=["date"]).copy()
    dates = sorted(pd.to_datetime(d["date"]).unique())
    idx = {dt:i for i,dt in enumerate(dates)}
    T=len(dates)
    out={}
    for m,g in d.groupby("mutation", sort=False):
        y=np.zeros(T, dtype=float)
        for _,r in g.iterrows():
            y[idx[pd.to_datetime(r["date"])]] = float(r["af"])
        out[m]=y
    return dates,out

def linspace_to_sector(n_items: int, sector_size: float):
    """Centers for n_items evenly across [0, sector_size)."""
    if n_items <= 0:
        return np.array([], dtype=float), 0.0
    step = sector_size / n_items
    return (np.arange(n_items, dtype=float) + 0.5) * step, step

def present_mask(y: np.ndarray, thr: float) -> np.ndarray:
    return (y >= thr)

def pearson_r(a: np.ndarray, b: np.ndarray) -> float:
    # both vectors must be same length & have variance
    if a.size != b.size: return np.nan
    if np.nanstd(a) == 0 or np.nanstd(b) == 0: return np.nan
    return float(np.corrcoef(a, b)[0,1])

# ---------------- Renderer ----------------
def circos_for_site(site):
    dsite = snv_g[snv_g["site_id"]==site]
    if dsite.empty:
        print(f"[skip] {site}")
        return
    dates,series = site_series(dsite)
    T=len(dates)
    if T==0:
        print(f"[skip] {site} (no dates)")
        return

    # per-lineage exact mutation list (only those present in H)
    lin2muts_exact = {}
    for lin in LINEAGES:
        muts = sig.loc[sig["lineage"]==lin, "mutation"].unique().tolist()
        muts = [m for m in muts if m in H.index]
        # sort by activation (desc) so bars are ordered
        if muts:
            muts = list(H.loc[muts, "activation"].sort_values(ascending=False).index)
        lin2muts_exact[lin] = muts

    # build alias maps per lineage (for alias chords)
    lin2alias_to_exact = {}
    for lin, muts in lin2muts_exact.items():
        amap = {}
        for m in muts:
            ak = alias_key(m)
            amap.setdefault(ak, []).append(m)
        lin2alias_to_exact[lin] = amap

    # Equal-size sectors (so all “boxes” look the same)
    SECTOR_SIZE = 100.0
    sector_sizes = {lin: SECTOR_SIZE for lin in LINEAGES}

    circ = Circos(sector_sizes, space=8)
    circ.inner_radius = R_INNER_HOLE  # IMPORTANT: 0..100 units

    # Cache bar-center positions for each (lin, mutation) for fast chord anchoring
    pos_center = {}  # (lin, mut) -> (center_x, slot_width)

    # draw sectors
    for sector in circ.sectors:
        lin = sector.name
        color = LINEAGE_COLOR.get(lin, "#333333")
        muts_lin = lin2muts_exact[lin]
        L = sector.size  # = SECTOR_SIZE

        # --- Outer track (scatter AF all muts, 93–100) ---
        tr_ts = sector.add_track((R_SCAT_IN, R_SCAT_OUT), r_pad_ratio=0.01)
        tr_ts.axis()
        if T > 0:
            xt = (np.arange(T, dtype=float) + 0.5) * (L / T)
        else:
            xt = np.array([], dtype=float)

        for m in muts_lin:
            y = series.get(m)
            if y is None or not np.any(y > 0):
                continue
            tr_ts.scatter(xt, y, s=10, vmin=0.0, vmax=1.0,
                          color=MUT_COLOR.get(m, "#666"), alpha=0.70)

        sector.text(lin, r=R_SCAT_OUT+2, size=12, color=color, weight="bold")

        if T >= 6:
            step = max(1, T//12)
            idx = np.arange(0, T, step, dtype=int)
            labels = [pd.to_datetime(dates[t]).strftime("%b") for t in idx]
            tr_ts.xticks((idx + 0.5) * (L / max(T,1)), labels, label_size=8, tick_length=0.8)

        # --- Inner track (NNMF bars ALL muts, 68–92) ---
        tr_bar = sector.add_track((R_BAR_IN, R_BAR_OUT), r_pad_ratio=0.01)
        tr_bar.axis()
        nM = len(muts_lin)
        if nM > 0:
            xbar, step_m = linspace_to_sector(nM, L)
            vals = H.reindex(muts_lin)["activation"].fillna(0.0).to_numpy(float)
            vmax = max(vals.max(), 1e-12)
            vals = vals / vmax
            tr_bar.bar(xbar, vals,
                       color=[MUT_COLOR.get(m, "#666") for m in muts_lin],
                       lw=0.0, alpha=0.98)
            for i, m in enumerate(muts_lin):
                pos_center[(lin, m)] = (xbar[i], step_m)

    # -------------------- Chords Layer A: shared-alias --------------------
    alias_to_lineages = {}
    for lin, amap in lin2alias_to_exact.items():
        for ak in amap.keys():
            alias_to_lineages.setdefault(ak, set()).add(lin)

    alias_links_made = 0
    for ak, lins in alias_to_lineages.items():
        if len(lins) < 2:
            continue

        # example exact mutation → color
        example_exact = next((m for m in ALL_MUTS if alias_key(m) == ak), None)
        chord_color = MUT_COLOR.get(example_exact, ALIAS_FALLBACK_COLOR)

        # link all lineage pairs that share this alias
        for a, b in combinations(sorted(lins, key=lambda x: LINEAGES.index(x)), 2):
            if alias_links_made >= MAX_ALIAS_LINKS_PER_PAIR * 6:  # global sanity cap
                break
            sa = next(s for s in circ.sectors if s.name == a)
            sb = next(s for s in circ.sectors if s.name == b)
            muts_a = lin2muts_exact[a]
            muts_b = lin2muts_exact[b]
            if not muts_a or not muts_b:
                continue
            # pick the first exact mapping for this alias in each lineage
            ma = lin2alias_to_exact[a][ak][0]
            mb = lin2alias_to_exact[b][ak][0]
            if (a,ma) not in pos_center or (b,mb) not in pos_center:
                continue
            xa, wa = pos_center[(a, ma)]
            xb, wb = pos_center[(b, mb)]
            startA = xa - 0.4*wa; endA = xa + 0.4*wa
            startB = xb - 0.4*wb; endB = xb + 0.4*wb
            circ.link((a, startA, endA),
                      (b, startB, endB),
                      r1=R_ALIAS_IN, r2=R_ALIAS_OUT,
                      fc=chord_color, ec="none", alpha=CHORD_ALPHA)
            alias_links_made += 1

    # -------------------- Chords Layer B: AF co-activation (correlation) --------------------
    corr_links_made = 0
    for a, b in combinations(LINEAGES, 2):
        muts_a = lin2muts_exact[a]
        muts_b = lin2muts_exact[b]
        if not muts_a or not muts_b:
            continue

        pairs = []
        for ma in muts_a:
            ya = series.get(ma)
            if ya is None: 
                continue
            mask_a = present_mask(ya, AF_PRESENT_THRESH)
            if mask_a.mean() < MIN_PRESENT_FRAC:
                continue
            for mb in muts_b:
                yb = series.get(mb)
                if yb is None:
                    continue
                mask_b = present_mask(yb, AF_PRESENT_THRESH)
                if mask_b.mean() < MIN_PRESENT_FRAC:
                    continue
                # overlap where either is present (to avoid spurious zeros)
                both = (mask_a | mask_b)
                if both.sum() < MIN_COMMON_DATES:
                    continue
                r = pearson_r(ya[both], yb[both])
                if np.isfinite(r) and r >= CORR_THRESHOLD:
                    pairs.append((r, ma, mb))

        # take top-K by correlation
        pairs.sort(key=lambda t: t[0], reverse=True)
        pairs = pairs[:MAX_CORR_LINKS_PER_PAIR]

        for r, ma, mb in pairs:
            if (a,ma) not in pos_center or (b,mb) not in pos_center:
                continue
            xa, wa = pos_center[(a, ma)]
            xb, wb = pos_center[(b, mb)]
            startA = xa - 0.35*wa; endA = xa + 0.35*wa
            startB = xb - 0.35*wb; endB = xb + 0.35*wb
            # scale alpha by correlation
            alpha = CHORD_ALPHA * (0.6 + 0.4 * float(r))  # 0.6..1.0 of CHORD_ALPHA
            circ.link((a, startA, endA),
                      (b, startB, endB),
                      r1=R_CORR_IN, r2=R_CORR_OUT,
                      fc=CORR_COLOR, ec="none", alpha=alpha)
            corr_links_made += 1

    # --- Title & legend ---
    fig = circ.plotfig(figsize=FIGSIZE)
    fig.suptitle(f"{site}\nAlias chords: {alias_links_made}  |  Corr chords: {corr_links_made} (r≥{CORR_THRESHOLD})",
                 y=0.97, fontsize=16, color="#002147", fontweight="bold")

    # Legend (mutations palette) — keep bottom clear
    show_muts = []
    for lin in LINEAGES:
        show_muts.extend(lin2muts_exact[lin])
    seen = set(); ordered = []
    for m in show_muts:
        if m not in seen:
            seen.add(m); ordered.append(m)
    ordered = ordered[:MAX_LEGEND]

    handles=[Line2D([0],[0],marker="o",ls="",color=MUT_COLOR.get(m,"#666"),markersize=6) for m in ordered]
    labels=ordered
    plt.subplots_adjust(bottom=0.26)
    leg1 = fig.legend(handles, labels,
               loc="lower center", bbox_to_anchor=(0.5, -0.14),
               ncol=min(20, max(2, int(np.ceil(len(labels)/2)))),
               frameon=True, fancybox=True, framealpha=0.95,
               borderpad=0.7, handletextpad=0.5, columnspacing=1.0, fontsize=7.2)

    # Small chord legend line
    proxy_alias = Line2D([0],[0], lw=6, color=ALIAS_FALLBACK_COLOR, alpha=CHORD_ALPHA)
    proxy_corr  = Line2D([0],[0], lw=6, color=CORR_COLOR,           alpha=CHORD_ALPHA)
    fig.legend([proxy_alias, proxy_corr], ["Shared-signature alias", "AF co-activation (corr)"],
               loc="lower center", bbox_to_anchor=(0.5, -0.04),
               ncol=2, frameon=False, fontsize=9)

    out = OUT_DIR / f"circos_{safe(site)}.png"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"[ok] {site} → {out}")

# ---------------- Run ----------------
for s in SITES:
    circos_for_site(s)
print(f"\n✅ Lineage-level Circos saved to {OUT_DIR}")


[ok] BE - Laupen - ARA Sensetal → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_BE___Laupen___ARA_Sensetal.png
[ok] GR - Chur - ARA Chur → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_GR___Chur___ARA_Chur.png
[ok] SG - Altenrhein - ARA Altenrhein → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_SG___Altenrhein___ARA_Altenrhein.png
[ok] TI - Lugano - CDA Lugano → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_TI___Lugano___CDA_Lugano.png
[ok] VD - Lausanne - STEP Vidy → C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\eda\nnmf_poisson\circos_lineage_chords\circos_VD___Lausanne___STEP_Vidy.png
[ok] ZH - ZÃ¼rich - ARA WerdhÃ¶lzli → C:\Users\osoom\OneDrive\Desktop\ssssss